# Proyecto final · Compras Claras — una ruta para todos
**Universidad Central · Big Data 2026-2S · alcance S01–S14 · versión de revisión docente**

**Producto:** clasificación administrativa de modalidad SECOP II y expediente verificable. Se usan dos etiquetas reales: Contratación directa y Contratación régimen especial. Esto NO detecta fraude.

**Recursos de estudiante (dos):** [Explorer interactivo y guía](../Presentaciones/proyecto-final-explorer.html) + este cuaderno. [Plan/rúbrica docente](../docs/PROYECTO_FINAL_S14_2026_PLAN.md).

**Entornos:**
- **Codespaces S13:** ejecutar E0–E4 y E7. No usar el perfil raíz Airflow. Reutiliza el laboratorio real de S13.
- **Databricks Free Edition:** importar este mismo cuaderno o copiar únicamente las celdas E5–E6; cargar secop_final.csv por la interfaz, y ejecutar E5–E6. NO ejecutar E0–E4 en Databricks.
- **Sin cuenta paga:** el entrenamiento Spark MLlib no está probado todavía en la cuenta Free del grupo. El docente debe hacer piloto real antes de calificar esta parte.

**Regla de aprendizaje:** no entregar resultados imaginarios. Un error de API/servicio se registra como error. El conjunto de entrenamiento es el CSV versionado del S13, no la muestra temporal descargada en E1.

**No hay elección de tema, etiquetas, algoritmo ni conjunto de datos.**

## Ruta sin relojes
E0 prepara Codespaces → E1 demuestra adquisición paralela → E2 genera CSV fijo train/test → E3 Dask procesa → E4 Dask reparte entrenamientos independientes → E5 Databricks integra con Spark SQL/Delta → E6 Spark MLlib entrena sobre particiones → E7 manifiesto + PR.

Los tiempos de clase y la entrega definitiva corresponden al docente, no a este cuaderno. Antes de realizar una acción, consulta el paso correspondiente del Explorer.

## E0 · Abre el entorno de S13 y comprueba archivos
**HAZ ESTO:** crea Codespace con la configuración S13, no con la configuración por defecto Airflow. Abre este cuaderno y ejecuta la celda.

**DEBE VERSE:** raíz del repositorio, archivo de 1.000 filas y biblioteca de S13 accesible.

**SI FALLA:** comprueba que tu Codespace tiene el repositorio completo y seleccionaste el kernel Python de S13; ejecuta python Laboratorios/S13_Codespaces/verificar_entorno.py en su terminal.

**EVIDENCIA:** fuente real encontrada, no un conjunto sintético.

**Cómo se lee:** la ruta identifica la fuente que se utilizará en todo el proyecto.
**Qué nos dice:** el laboratorio se puede continuar con los datos comunes.
**Qué NO permite concluir:** que los modelos ya están entrenados.
**Qué error común:** abrir el cuaderno en otro entorno sin los archivos del repositorio.

In [ ]:
from pathlib import Path
import sys, os, hashlib, json, importlib.util
raiz = Path.cwd().resolve()
for candidato in [raiz, *raiz.parents]:
    if (candidato / "Laboratorios/S13_Codespaces/analisis.py").exists():
        raiz = candidato
        break
else:
    raise FileNotFoundError("Abre este cuaderno en Codespaces del repo BigData2026, perfil S13.")
sys.path.insert(0, str(raiz / "Laboratorios/S13_Codespaces"))
import analisis as S13
directorio = raiz / "Laboratorios/Proyecto_Final_2026/salidas"
directorio.mkdir(parents=True, exist_ok=True)
assert S13.CSV.exists(), "Falta el CSV real de S13"
print("RAIZ:", raiz)
print("FUENTE:", S13.CSV.relative_to(raiz))
print("CPU lógicas visibles:", os.cpu_count())
print("CSV SHA256:", hashlib.sha256(S13.CSV.read_bytes()).hexdigest())
print("OK: material de S13 localizado.")

## E1 · Descarga paralela real, acotada y sin simulaciones
**HAZ ESTO:** realiza las mismas dos consultas consecutivas (20 registros por página) de forma secuencial y con 2 workers. Se usa el endpoint público SECOP II y orden por identificador interno Socrata. Esta descarga NO reemplaza el snapshot congelado de S13.

**DEBE VERSE:** filas recuperadas, un hash de contenido y comparación de tiempos; si HTTP 429 o la fuente falla, un registro del error (NO afirmar éxito).

**SI FALLA:** conserva A_descarga.json con estado y mensaje. La red pública puede cambiar o limitar peticiones; el resto del taller trabaja con los 1.000 registros preparados. Si el servicio varía entre las dos solicitudes, reportar discrepancia.

**EVIDENCIA:** A_descarga.json.

**Cómo se lee:** tiempos totales medidos para 2 páginas y un hash canónico por método.
**Qué nos dice:** si las respuestas equivalen y si hubo concurrencia real.
**Qué NO permite concluir:** que dos workers siempre descargan más rápido o que se obtuvo un snapshot inmutable del API.
**Qué error común:** comparar cantidades distintas o duplicar páginas por un offset mal calculado.

In [ ]:
# E1 · Código visible. Máximo 2 páginas × 20 filas × 2 métodos: 4 consultas iniciales.
# Instala requests sólo si falta en el Codespace.
if importlib.util.find_spec("requests") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests>=2.31,<3"])
import requests, time
from concurrent.futures import ThreadPoolExecutor

URL_SECOP = "https://www.datos.gov.co/resource/p6dx-8zbt.json"
OFFSETS = [0, 20]
PARAMETROS = {"$limit": 20, "$order": ":id ASC"}
def pagina_secop(offset):
    parametros = dict(PARAMETROS, **{"$offset": offset})
    for intento in (1, 2):
        try:
            respuesta = requests.get(URL_SECOP, params=parametros,
                                     headers={"User-Agent":"UC-BigData2026-final-educativo/1.0",
                                              "Accept":"application/json"},
                                     timeout=25)
            if respuesta.status_code == 429:
                if intento == 2: respuesta.raise_for_status()
                pausa = respuesta.headers.get("Retry-After", "2")
                time.sleep(min(15, max(1, int(pausa))) if pausa.isdigit() else 3)
                continue
            respuesta.raise_for_status()
            filas = respuesta.json()
            assert isinstance(filas, list) and len(filas) <= 20
            return offset, filas
        except requests.RequestException:
            if intento == 2: raise
            time.sleep(2)
    raise RuntimeError("Sin respuesta de API")

def medir_descarga(concurrente):
    inicio = time.perf_counter()
    if concurrente:
        with ThreadPoolExecutor(max_workers=2) as pool:
            paginas = list(pool.map(pagina_secop, OFFSETS))
    else:
        paginas = [pagina_secop(o) for o in OFFSETS]
    filas = [r for _, bloque in sorted(paginas) for r in bloque]
    canon = json.dumps(filas, sort_keys=True, ensure_ascii=False, separators=(",",":"))
    return {"filas":len(filas), "hash":hashlib.sha256(canon.encode()).hexdigest(),
            "segundos":round(time.perf_counter()-inicio,4), "paginas":len(paginas)}

try:
    serie = medir_descarga(False)
    paralelo = medir_descarga(True)
    evidencia_a = {"exito":True, "modo":"API real", "serie":serie,
                   "paralelo":paralelo,
                   "misma_respuesta":serie["hash"]==paralelo["hash"],
                   "nota":"Fuente mutable; la igualdad puede fallar por actualizaciones."}
except Exception as error:
    evidencia_a = {"exito":False, "modo":"API real", "error_tipo":type(error).__name__,
                   "detalle":str(error)[:350],
                   "nota":"Falla registrada; E2 usa el snapshot congelado existente."}
(directorio/"A_descarga.json").write_text(
    json.dumps(evidencia_a,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps(evidencia_a,indent=2,ensure_ascii=False))

## E2 · Preparar exactamente los mismos datos para Dask y Databricks
**HAZ ESTO:** filtra las dos modalidades ya usadas en S13, elimina filas repetidas por identificador y texto vacío. Reutiliza el split estratificado semilla 42 de S13. La columna split será la fuente de verdad en Spark.

**DEBE VERSE:** dos modalidades, IDs únicos y archivos secop_final.csv / B_fuente.json.

**SI FALLA:** verifica los nombres reales de columnas de S13; NO cambies etiquetas ni sustituyas por datos sintéticos.

**EVIDENCIA:** CSV y manifiesto con SHA-256.

**Cómo se lee:** train son las observaciones para ajustar el modelo; test son registros reservados.
**Qué nos dice:** ambos motores podrán ver exactamente la misma prueba.
**Qué NO permite concluir:** que un modelo tendrá buen rendimiento o que las etiquetas miden riesgo.
**Qué error común:** ajustar el TF-IDF usando test antes de separar los datos.

In [ ]:
import pandas as pd
tabla = S13.leer_etiquetados()       # verifica 1000 filas reales
muestra = S13.preparar_clasificacion(tabla)
x_train,x_test,y_train,y_test = S13.separar_muestra(tabla)
splits = pd.Series("test",index=muestra.index)
splits.loc[x_train.index]="train"
base = pd.DataFrame({
    "id_del_proceso":muestra["id_del_proceso"].astype(str),
    "texto":muestra[S13.COL_TEXTO].astype(str),
    "modalidad":muestra[S13.COL_LABEL].astype(str),
    "split":splits
})
assert base["id_del_proceso"].is_unique
assert set(base["modalidad"])==set(S13.CLASES)
assert set(base["split"])=={"train","test"}
assert base.loc[base.split=="train","id_del_proceso"].isin(
       base.loc[base.split=="test","id_del_proceso"]).sum()==0
ruta_csv=directorio/"secop_final.csv"
base.to_csv(ruta_csv,index=False,encoding="utf-8")
evidencia_b={"fuente":str(S13.CSV.relative_to(raiz)),
 "sha256_fuente":hashlib.sha256(S13.CSV.read_bytes()).hexdigest(),
 "csv_salida":"secop_final.csv","sha256_salida":hashlib.sha256(ruta_csv.read_bytes()).hexdigest(),
 "filas_originales":int(len(tabla)),"filas_modelo":int(len(base)),
 "train":int((base.split=="train").sum()),
 "test":int((base.split=="test").sum()),
 "clases":base.modalidad.value_counts().to_dict(),"random_state":42}
(directorio/"B_fuente.json").write_text(json.dumps(
    evidencia_b,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(evidencia_b,indent=2,ensure_ascii=False))

## E3 · Dask: procesamiento real de los mismos registros
**HAZ ESTO:** usa 2 workers de Dask Distributed con 1 hilo cada uno; calcula conteos por modalidad y comprueba la igualdad contra pandas. Los workers pueden ser hilos dentro de un mismo Codespace: no son computadores físicos diferentes.

**DEBE VERSE:** 2 workers observados y los mismos conteos.

**SI FALLA:** comprueba dask.distributed instalado y carga del entorno S13. No aumentes workers para intentar esconder errores.

**EVIDENCIA:** C_dask.json. El tiempo no es una calificación.

**Cómo se lee:** workers_reales es el número reconocido por el scheduler; particiones son bloques de datos.
**Qué nos dice:** Dask calculó el mismo resultado desde tareas particionadas.
**Qué NO permite concluir:** que el problema necesita un clúster o que Dask es siempre más rápido.
**Qué error común:** decir que Dask ejecutó al crear el grafo sin llamar compute().

In [ ]:
import dask.dataframe as dd
from distributed import Client
import time
inicio=time.perf_counter()
with Client(processes=False,n_workers=2,threads_per_worker=1,dashboard_address=None) as cliente:
    workers=cliente.scheduler_info()["workers"]
    ddf=dd.from_pandas(base,npartitions=4)
    conteos_dask={str(k):int(v) for k,v in
                  ddf.groupby("modalidad").size().compute().items()}
    trabajadores=[{"id":str(k),"hilos":int(v["nthreads"])}
                   for k,v in workers.items()]
tiempo=time.perf_counter()-inicio
conteos_pandas={str(k):int(v) for k,v in base.groupby("modalidad").size().items()}
assert conteos_dask==conteos_pandas and len(trabajadores)==2
evidencia_c={"workers_reales":trabajadores,"particiones":4,
 "conteos_dask":conteos_dask,"conteos_pandas":conteos_pandas,
 "coinciden":True,"segundos_con_inicio":round(tiempo,5),
 "nota":"Dask DataFrame distribuido; muestra pequeña sin garantía de aceleración."}
(directorio/"C_dask.json").write_text(json.dumps(
    evidencia_c,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(evidencia_c,indent=2,ensure_ascii=False))

## E4 · Entrenamiento de tres modelos independientes con Dask
**HAZ ESTO:** aprovecha el laboratorio S13. Entrena regresión logística con TF-IDF, valores de C iguales para todos: 0.1, 1.0 y 10.0. La validación cruzada usa solamente train. El test se utiliza una vez al final.

**DEBE VERSE:** tres valores de C y sus F1, dos workers observados y una F1 final. Contrasta contra el clasificador de referencia DummyClassifier.

**SI FALLA:** usa la muestra real original, revisa memoria y comprueba el módulo S13. No ejecutes benchmarks de 50.000.

**EVIDENCIA:** D_modelo_dask.json.

**Cómo se lee:** F1 armoniza precisión y cobertura; la métrica representa estas dos clases y este test.
**Qué nos dice:** si el clasificador supera o no al control sencillo.
**Qué NO permite concluir:** generalización a cualquier contrato futuro, ni entrenar un único modelo repartido internamente.
**Qué error común:** elegir el hiperparámetro usando el test.

In [ ]:
referencia=S13.entrenar_y_evaluar(tabla)
resultado_dask=S13.experimentos_distribuidos(tabla,n_workers=2)
assert len(resultado_dask["configuraciones"])==3
evidencia_d={"f1_referencia":referencia["f1_macro"],
 "f1_dummy":referencia["f1_control"],
 "matriz_confusion":referencia["matriz_confusion"],
 "workers":resultado_dask["workers_en_un_codespace"],
 "configuraciones":resultado_dask["configuraciones"],
 "mejor_C":resultado_dask["mejor_C"],
 "f1_prueba_final":resultado_dask["f1_prueba_final"],
 "significado":"Dask coordina 3 fits independientes; NO distribuye internamente un fit()."}
(directorio/"D_modelo_dask.json").write_text(json.dumps(
    evidencia_d,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(evidencia_d,indent=2,ensure_ascii=False))

## E5 · Databricks Free: archivo local → tabla Delta → Spark SQL
**CAMBIA DE ENTORNO.** Descarga secop_final.csv del Codespace y súbelo manualmente en Databricks Free mediante **Catalog → Create/Modify table from upload**. El menú puede cambiar; busca la carga de CSV. No intentes descargar desde SECOP en Databricks, porque el acceso saliente está restringido.

1. Crea una tabla Delta a partir del CSV. Conserva nombres id_del_proceso, texto, modalidad, split.
2. Abre este mismo cuaderno en Databricks y **ejecuta E5/E6 solamente**. Debes editar TABLA_SUBIDA con el nombre real en tu entorno (catálogo.esquema.tabla). No es una elección académica.
3. La celda crea tablas Bronze, Silver, Gold con Spark SQL/Delta sin añadir tecnologías no vistas.
4. La tabla Gold muestra cuántos procesos hay por clase y partición; deben concordar con E2.

**DEBE VERSE:** 3 tablas, consulta, conteos y JSON E_databricks.
**SI FALLA:** verifica nombre real de catálogo/esquema, carga CSV y cuotas del plan. Comunica el fallo al docente antes de concluir.
**EVIDENCIA:** salida JSON E_databricks, captura real y consulta SQL.

**Cómo se lee:** Bronze preserva origen; Silver valida columnas; Gold agrega.
**Qué nos dice:** los registros pueden procesarse y consultarse en plataforma Spark.
**Qué NO permite concluir:** que se construyó un clúster personalizado ni que se realizó entrenamiento.
**Qué error común:** usar rutas relativas del Codespace en Databricks o volver a dividir train/test.

In [ ]:
# E5 · EJECUTAR SOLO EN UN NOTEBOOK DATABRICKS DESPUÉS DE SUBIR EL CSV.
# Sustituye únicamente el nombre de TABLA_SUBIDA por el que muestra Catalog.
TABLA_SUBIDA = "workspace.default.secop_final"
if "dbutils" not in globals():
    print("E5 NO EJECUTADO: este bloque corresponde a Databricks, no a Codespaces.")
else:
    from pyspark.sql import functions as F
    import json
    datos_spark=spark.table(TABLA_SUBIDA)
    obligatorias={"id_del_proceso","texto","modalidad","split"}
    assert obligatorias.issubset(set(datos_spark.columns)), "Revisa encabezados del CSV subido"
    prefijo=TABLA_SUBIDA.rsplit(".",1)[0]
    BRONZE=prefijo+".uc_final_bronze"
    SILVER=prefijo+".uc_final_silver"
    GOLD=prefijo+".uc_final_gold"
    datos_spark.write.format("delta").mode("overwrite").saveAsTable(BRONZE)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER} USING DELTA AS
    SELECT CAST(id_del_proceso AS STRING) AS id_del_proceso,
           CAST(texto AS STRING) AS texto,
           CAST(modalidad AS STRING) AS modalidad,
           CAST(split AS STRING) AS split
    FROM {BRONZE}
    WHERE texto IS NOT NULL AND TRIM(texto) <> ''
      AND split IN ('train','test')
      AND modalidad IN ('Contratación directa','Contratación régimen especial')
    """)
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD} USING DELTA AS
    SELECT modalidad,split,COUNT(*) AS registros
    FROM {SILVER} GROUP BY modalidad,split
    """)
    assert spark.table(SILVER).count()==datos_spark.count(), "E5: se perdieron filas"
    assert spark.table(SILVER).select("id_del_proceso").distinct().count()==datos_spark.count(), "E5: IDs duplicados"
    filas_gold=[r.asDict() for r in spark.table(GOLD).orderBy("modalidad","split").collect()]
    evidencia_e={"ejecucion":"DATABRICKS_REAL","tabla_subida":TABLA_SUBIDA,
                 "bronze":BRONZE,"silver":SILVER,"gold":GOLD,
                 "filas":datos_spark.count(),"conteos":filas_gold}
    print("E_DATABRICKS_JSON="+json.dumps(evidencia_e,ensure_ascii=False))
    display(spark.table(GOLD))

## E6 · Entrenamiento Spark MLlib guiado: mismo caso y mismo split
**Este bloque no se introdujo como práctica principal en S14:** el código está resuelto para que la exigencia evalúe la integración y lectura de evidencias, no memorizar una API nueva.

Pipeline: Tokenizer separa palabras → HashingTF crea características numéricas → IDF reduce peso de términos frecuentes → LogisticRegression ajusta una clasificación binaria. La representación difiere del TfidfVectorizer de scikit-learn, por lo que NO se exige igualdad exacta de F1 entre motores. Sí se exige **el mismo conjunto test**, que ya viene etiquetado en la columna split.

**HAZ ESTO:** después de E5, ejecuta en Databricks. No renombres las dos etiquetas ni vuelvas a dividir los datos.

**DEBE VERSE:** predicciones, F1 macro y matriz de confusión en test. **SI FALLA:** revisar entorno, tabla Silver, permisos o cuota del servicio. **EVIDENCIA:** F_spark_modelo como salida estructurada.

**Cómo se lee:** la matriz distingue modalidad real frente a predicha.
**Qué nos dice:** el comportamiento de este clasificador con estos registros.
**Qué NO permite concluir:** que predice fraude ni que 1.000 filas demuestran escalabilidad masiva.
**Qué error común:** entrenar sobre test o afirmar que los dos algoritmos usan características idénticas.

In [ ]:
# E6 · EJECUTAR SOLO EN DATABRICKS DESPUÉS DE E5.
if "dbutils" not in globals():
    print("E6 NO EJECUTADO: requiere el cuaderno remoto Databricks y E5 correcto.")
else:
    from pyspark.sql import functions as F
    from pyspark.ml import Pipeline
    from pyspark.ml.feature import Tokenizer, HashingTF, IDF
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator
    datos_ml=(spark.table(SILVER)
      .withColumn("label",F.when(F.col("modalidad")=="Contratación directa",F.lit(0.0)).otherwise(F.lit(1.0))))
    train_ml=datos_ml.filter(F.col("split")=="train")
    test_ml=datos_ml.filter(F.col("split")=="test")
    assert train_ml.count()>0 and test_ml.count()>0
    tokenizador=Tokenizer(inputCol="texto",outputCol="tokens")
    tf=HashingTF(inputCol="tokens",outputCol="tf",numFeatures=4096)
    idf=IDF(inputCol="tf",outputCol="features",minDocFreq=2)
    logistica=LogisticRegression(featuresCol="features",labelCol="label",maxIter=25,regParam=.1)
    pipeline=Pipeline(stages=[tokenizador,tf,idf,logistica])
    modelo_spark=pipeline.fit(train_ml)   # entrenamiento con cálculos de Spark sobre particiones
    pred=modelo_spark.transform(test_ml)
    conteos={(float(r["label"]),float(r["prediction"])):int(r["count"])
       for r in pred.groupBy("label","prediction").count().collect()}
    f1_por_clase=[]
    for cls in (0.0,1.0):
        vp=conteos.get((cls,cls),0)
        fp=sum(n for (real,estim),n in conteos.items() if estim==cls and real!=cls)
        fn=sum(n for (real,estim),n in conteos.items() if real==cls and estim!=cls)
        f1_por_clase.append((2*vp/(2*vp+fp+fn)) if 2*vp+fp+fn else 0.)
    F1_MACRO=round(sum(f1_por_clase)/2,4)
    pred.select("id_del_proceso","label","prediction").write.format("delta").mode("overwrite").saveAsTable(
        prefijo+".uc_final_predicciones")
    evidencia_f={"ejecucion":"SPARK_MLLIB_REAL","entrenamiento":train_ml.count(),
      "prueba":test_ml.count(),"f1_macro":F1_MACRO,
      "f1_por_clase":[round(x,4) for x in f1_por_clase],
      "matriz_real_0_1_pred_0_1":[
         [conteos.get((0.0,0.0),0),conteos.get((0.0,1.0),0)],
         [conteos.get((1.0,0.0),0),conteos.get((1.0,1.0),0)]],
      "tabla_predicciones":prefijo+".uc_final_predicciones",
      "limitacion":"No equivale numéricamente a sklearn; representa clases administrativas."}
    print("F_SPARK_MODELO_JSON="+json.dumps(evidencia_f,ensure_ascii=False))
    display(pred.select("id_del_proceso","modalidad","prediction").limit(10))

## E7 · Manifest final en Codespaces, PR y microdefensa
**VUELVE A CODESPACES.** E7 registra lo que **sí existe**; no marca como terminada una etapa Databricks sin evidencia real. Si E5/E6 se ejecutaron, incorpora sus JSON desde las salidas del notebook remoto y las capturas en el repositorio privado del equipo. El profesor verificará las tablas.

**HAZ ESTO:** ejecutar la celda, abrir manifest_final.json y comprobar cada ruta. Colocar los archivos producidos en talleres/Final/ del repositorio privado ya usado en TC1/TC2, hacer commit y PR al repositorio del equipo. No abrir PR hacia el repositorio público del docente.

**DEBE VERSE:** manifiesto con estados reales. **SI FALLA:** identifica qué evidencia falta y vuelve al bloque afectado.

**Cómo se lee:** presente o pendiente representa existencia de un archivo, NO aprobación automática.
**Qué nos dice:** qué recursos se pueden auditar.
**Qué NO permite concluir:** que un archivo prueba por sí solo autoría o que Databricks corrió correctamente.
**Qué error común:** copiar F1 o afirmar ejecución sin dejar traza verificable.

In [ ]:
# E7 · Ejecuta en Codespaces después de E0–E4.
esperados=["A_descarga.json","B_fuente.json","C_dask.json","D_modelo_dask.json",
           "secop_final.csv","E_databricks.json","F_spark_modelo.json"]
estado={}
for nombre in esperados:
    ruta=directorio/nombre
    estado[nombre]={"estado":"presente" if ruta.is_file() else "pendiente"}
    if ruta.is_file():
        estado[nombre]["sha256"]=hashlib.sha256(ruta.read_bytes()).hexdigest()
        estado[nombre]["bytes"]=ruta.stat().st_size
manifiesto={"curso":"BigData 2026-2S","taller":"Proyecto final Compras Claras",
 "mismo_caso_todos":True,"fuente_entrenamiento":"snapshot SECOP S13",
 "estados":estado,"nota":"Presencia no significa que la etapa esté validada.",
 "advertencia":"Sin detectar fraude. No exponer secretos ni identificadores privados."}
(directorio/"manifest_final.json").write_text(json.dumps(
    manifiesto,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(manifiesto,ensure_ascii=False,indent=2))

## Cierre · hoja de trucos del proyecto
| Herramienta | Función | Demostración real |
|---|---|---|
| ThreadPoolExecutor | varias solicitudes de red simultáneas | E1, máximo 2 workers |
| Parquet / CSV | persistencia y transporte de datos | E2 |
| Dask DataFrame | tareas/particiones y compute | E3 |
| Dask Distributed + scikit-learn | distribuye **tres entrenamientos independientes** | E4 |
| Spark SQL / Delta | transformación, agregación y tablas | E5 |
| Spark MLlib | entrena modelo con cálculo Spark | E6 |
| Explorer | consulta y simulación pedagógica | guía HTML |

**No se califica:** ser más rápido, comprar cómputo, repetir 50.000 filas, entrenar un LLM, instalar Atlas/Cassandra/Aura de nuevo.

**Se califica:** trabajo verdadero, datos trazables, resultados verificables, interpretación responsable, rutas reproducibles y defensa. Fechas y ponderación definitivas: pendientes de validación docente del grupo 2.